# Bangla Multimodal Fake News: Model Training

This notebook extracts features and trains the models. It needs a **T4 GPU**: Runtime > Change runtime type > T4 GPU.

Run `01_preprocessing.ipynb` first, so that `unified_split.csv` exists on Drive.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 1. Unpack the project code

In [ ]:
import shutil
import zipfile
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")
CODE_DIR = Path("/content/Bangla-Multimodal-FakeNews")

candidates = list(DRIVE_ROOT.glob("Bangla-Multimodal-FakeNews*.zip")) + list(
    (DRIVE_ROOT / "BFND-data").glob("Bangla-Multimodal-FakeNews*.zip")
)
assert candidates, "Code zip not found on Drive. Upload Bangla-Multimodal-FakeNews.zip first."
ZIP = max(candidates, key=lambda p: p.stat().st_mtime)
print("Using zip:", ZIP)

shutil.rmtree(CODE_DIR, ignore_errors=True)   # remove any old copy so stale code cannot run
with zipfile.ZipFile(ZIP) as z:
    z.extractall("/content")
print("Files in zip:", len(zipfile.ZipFile(ZIP).namelist()))

In [ ]:
import os
from pathlib import Path

os.chdir(CODE_DIR)
config_text = Path("configs/config.py").read_text(encoding="utf-8")
assert "CLIP_IMAGE_MODEL" in config_text, (
    "Old code detected. Upload the latest zip to Drive, then run the unpack cell again."
)
print("Code version OK. Working directory:", os.getcwd())

## 2. Check the GPU

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No GPU. Set Runtime > Change runtime type > T4 GPU, restart, and run again."
)
print("GPU:", torch.cuda.get_device_name(0))

## 3. Install packages

In [ ]:
!pip install -q "accelerate>=0.26" "transformers>=4.46" "sentence-transformers>=3.0" "rank-bm25>=0.2.2" joblib openpyxl

## 4. Preflight

In [ ]:
!python scripts/00_preflight.py

## 5. Feature extraction

Builds BanglaBERT embeddings and CLIP image and text embeddings. The cultural and evidence features are built only when their input files exist; otherwise a [SKIP] line is printed.

In [ ]:
!python scripts/02_extract_features.py

## 6. Check the feature files

Expected shapes: `bert (3910, 768)`, `clip_img (3910, 512)`, `clip_txt (3910, 512)`, `sim (3910, 1)`. The row count changes if the data changes.

In [ ]:
import numpy as np
import pandas as pd
from configs.config import FEATURE_DIR

meta = pd.read_csv(FEATURE_DIR / "meta.csv")
print(meta["split"].value_counts())
for name in ["bert", "clip_img", "clip_txt", "sim"]:
    print(name, np.load(FEATURE_DIR / f"{name}.npy").shape)

## 7. Length-only baseline

A model that sees only the text length. If it scores close to the real models, those models may be relying on length, so their results need careful reading.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

from configs.config import FEATURE_DIR

meta = pd.read_csv(FEATURE_DIR / "meta.csv")
meta["n_chars"] = meta["model_text"].astype(str).str.len()
X_len = np.log1p(meta[["n_chars"]].to_numpy(dtype=float))
train_mask = meta["split"].eq("train").to_numpy()
test_mask = meta["split"].eq("test").to_numpy()
y = meta["y"].astype(int).to_numpy()

clf = LogisticRegression(max_iter=1000, class_weight="balanced")
clf.fit(X_len[train_mask], y[train_mask])
pred = clf.predict(X_len[test_mask])
print("Length-only accuracy (test):", round((pred == y[test_mask]).mean(), 4))
print("Length-only macro-F1 (test):", round(f1_score(y[test_mask], pred, average="macro"), 4))

## 8. E1: fine-tune BanglaBERT (text only)

Runs three seeds. Expect a long run, roughly 20 to 40 minutes on a T4.

In [ ]:
!python scripts/03_train_text_baseline.py

## 9. Fusion experiments, decision layer and evidence check

In [ ]:
!python scripts/04_train_fusion.py

## 10. Summary

In [ ]:
from configs.config import RESULTS_DIR

print((RESULTS_DIR / "summary.md").read_text(encoding="utf-8"))